## 프로젝트 루트 설정

In [1]:
from pathlib import Path
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
data_dir = project_root / "data" / "raw"
print("프로젝트 루트:", project_root)
print("데이터 폴더:", data_dir)
print("데이터 폴더 존재:", data_dir.exists())

프로젝트 루트: c:\Dev\ai-data-analysis
데이터 폴더: c:\Dev\ai-data-analysis\data\raw
데이터 폴더 존재: True


## 36. pandas와 CSV불러오기

In [2]:
import pandas as pd

customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")

## 37. 기본 구조와 주요 키 확인

In [3]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}
for name, df in datasets.items():
    print(name, df.shape, df.columns.tolist())

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (301, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (765, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [4]:
key_checks = {
    "customers.customer_id": customers["customer_id"],
    "products.product_id": products["product_id"],
    "orders.order_id": orders["order_id"],
    "order_items.order_item_id": order_items["order_item_id"],
}
for name, series in key_checks.items():
    print(
        name,
        "결측:", series.isna().sum(),
        "중복:", series.duplicated().sum(),
    )

customers.customer_id 결측: 0 중복: 0
products.product_id 결측: 0 중복: 0
orders.order_id 결측: 0 중복: 0
order_items.order_item_id 결측: 0 중복: 0


## 38. Series와 DataFrame 선택

In [5]:
city_series = customers["city"]
customer_view = customers[
    ["customer_id", "gender", "age", "city"]
]
print(type(city_series))
print(type(customer_view))
display(customer_view.head())

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


## 단일 조건 필터링

In [6]:
customers_over_30 = customers[
    customers["age"] >= 30
]
print(len(customers), len(customers_over_30))
display(customers_over_30.head())

150 111


,customer_id,name,gender,age,city,signup_date
1,2,김정호,F,32,대구,2025-11-29
2,3,이경수,F,61,성남,2024-07-09
3,4,조영호,F,55,울산,2026-05-10
5,6,김지원,F,32,성남,2026-07-24
6,7,이상현,F,53,인천,2025-01-08


## 복합 조건 필터링

In [7]:
seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울")
]

display(seoul_over_30.head())

,customer_id,name,gender,age,city,signup_date
8,9,송지민,M,69,서울,2025-11-15
14,15,장정식,M,69,서울,2026-07-01
29,30,이민재,F,32,서울,2023-08-10
47,48,김예은,F,47,서울,2025-04-28
65,66,김재호,F,39,서울,2025-12-30


In [8]:
seoul_or_busan = customers[
    customers["city"].isin(["서울", "부산"])
]
display(
    seoul_or_busan["city"].value_counts()
)


city
부산    16
서울    15
Name: count, dtype: int64

In [9]:
not_completed = orders[
    ~(orders["order_status"] == "completed")
]
display(
    not_completed["order_status"].value_counts(
        dropna=False
    )
)

order_status
cancelled    65
refunded     52
Name: count, dtype: int64

## 41. 상품 가격 정렬

In [10]:
# sort 라고 표현한다 상품 가격을 정렬 한다 ascendig, descending ascending = 오름차순, descending = 내림차순
expensive_products = (
    products
    .sort_values("price", ascending=False)
    .head(10)
)
display(
    expensive_products[
        [
            "product_id",
            "product_name",
            "category",
            "price",
        ]
    ]
)

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


## 42. 작업용 복사본과 파생 컬럼

In [11]:
#원본을 보존하기 위해서는 copy()를 사용하여 복사본을 만들어서 작업하는 것이 좋다.
order_items_work = order_items.copy()

order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]
)

In [12]:
print(order_items_work.head())

   order_item_id  order_id  product_id  quantity  unit_price  line_total
0              1         1         100         3      102000      306000
1              2         1          87         5       25000      125000
2              3         1           7         3      142000      426000
3              4         1           9         3      193000      579000
4              5         2          72         4      189000      756000


결과확인:

In [13]:

display(
    order_items_work[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "line_total",
        ]
    ].head()

)

,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,100,3,102000,306000
1,2,1,87,5,25000,125000
2,3,1,7,3,142000,426000
3,4,1,9,3,193000,579000
4,5,2,72,4,189000,756000


## 43. 수작업 검증

In [14]:
# iloc 는 위치 기반 인덱싱을 의미한다. 0번째 행을 가져오고 싶으면 iloc[0] 을 사용하면 된다.
sample = order_items_work.iloc[0]
expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]
print("수작업:", expected)
print("파생 컬럼:", actual)
print("일치:", expected == actual)

수작업: 306000
파생 컬럼: 306000
일치: True


## 44. 전체 주문상세 금액

In [15]:
all_order_amount = order_items_work["line_total"].sum()
print("전체 주문상세 금액:", all_order_amount)

전체 주문상세 금액: 258932220


현재 합계에는 취소 또는 환불 주문이 포함될 수 있으므로,

완료 주문 기준 매출이 아니라 전체 주문상세 금액으로 표현한다.

## 45. 병합용 주문 컬럼 선택

In [16]:
orders_for_merge = orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status"
    ]
].copy()

print(orders_for_merge.shape)
print(orders_for_merge.head())

(301, 4)
   order_id  customer_id  order_date order_status
0         1          123  2026-06-03    completed
1         2           77  2025-08-19    cancelled
2         3          138  2025-12-16    cancelled
3         4           57  2026-02-26    cancelled
4         5          125  2026-01-17    cancelled


## 46. 주문상세와 주문 병합

In [17]:
order_sales = (
    order_items_work
    .merge(
        orders_for_merge,
        on="order_id",
        how="left",
        validate="many_to_one",
        indicator="order_match",
    )
)


## 47. 병합 검증

In [18]:

print("병합 전 행 수:", len(order_items_work))
print("병합 후 행 수:", len(order_sales))
display(
    order_sales["order_match"].value_counts(
        dropna=False
    )
)

병합 전 행 수: 765
병합 후 행 수: 765


order_match
both          764
left_only       1
right_only      0
Name: count, dtype: int64

미매칭 확인:

In [19]:
unmatched_orders = order_sales[
    order_sales["order_match"] != "both"
]
display(unmatched_orders.head())

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match
764,765,302,200,10,332222,3322220,NaN,NaN,NaN,left_only


## 48. 완료 주문 분석셋

In [20]:
display(
    order_sales["order_status"].value_counts(
        dropna=False
    )
)

order_status
completed    474
cancelled    162
refunded     128
NaN            1
Name: count, dtype: int64

In [21]:
completed_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

In [22]:
print("완료 주문상세 행:", len(completed_sales))
print(
    "완료 주문 수:",
    completed_sales["order_id"].nunique(),
)
print(
    "완료 주문 고객 수:",
    completed_sales["customer_id"].nunique(),
)
print(
    "완료 주문 매출:",
    completed_sales["line_total"].sum(),
)


완료 주문상세 행: 474
완료 주문 수: 184
완료 주문 고객 수: 100
완료 주문 매출: 148990000


## 49. 필요한 상품 정보만 선택

In [23]:

products_for_merge = products[
    [
        "product_id",
        "product_name",
        "category",
    ]

].copy()

## 50. 완료 주문상세와 상품 병합

In [24]:
completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)
print(completed_items.shape)
print(completed_items.head())

(474, 13)
   order_item_id  order_id  product_id  quantity  unit_price  line_total  \
0              1         1         100         3      102000      306000   
1              2         1          87         5       25000      125000   
2              3         1           7         3      142000      426000   
3              4         1           9         3      193000      579000   
4             13         6          83         3       24000       72000   

   customer_id  order_date order_status order_match product_name category  \
0        123.0  2026-06-03    completed        both    도서 상품 100       도서   
1        123.0  2026-06-03    completed        both    도서 상품 087       도서   
2        123.0  2026-06-03    completed        both    도서 상품 007       도서   
3        123.0  2026-06-03    completed        both   스포츠 상품 009      스포츠   
4         87.0  2026-04-17    completed        both  전자기기 상품 083     전자기기   

  product_match  
0          both  
1          both  
2          both 

In [25]:
print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

474 474


product_match
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 51. 카테고리별 매출

In [26]:
category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
3,스포츠,31743000,85,67,295,100
5,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
1,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42
0,도서,16389000,52,46,149,58
6,패션,10587000,33,27,111,37


## 52. 카테고리 합계 검증

In [27]:
category_total = category_sales["total_sales"].sum()
completed_total = completed_items["line_total"].sum()
print(category_total)
print(completed_total)
print(category_total == completed_total)

148990000
148990000
True


## 53. 상품별 매출

In [28]:

product_sales = (
    completed_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_sales=("line_total", "sum"),
        quantity_sold=("quantity", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
    )
    .sort_values("total_sales", ascending=False)
)
display(product_sales.head(10))

,product_id,product_name,category,total_sales,quantity_sold,order_count,customer_count
39,41,스포츠 상품 041,스포츠,5705000,35,12,11
11,12,식품 상품 012,식품,4375000,25,7,7
8,9,스포츠 상품 009,스포츠,3860000,20,6,5
70,72,뷰티 상품 072,뷰티,3780000,20,6,6
69,71,전자기기 상품 071,전자기기,3703000,23,5,5
66,68,스포츠 상품 068,스포츠,3640000,26,8,8
78,81,전자기기 상품 081,전자기기,3630000,22,6,6
10,11,패션 상품 011,패션,3565000,31,7,7
20,22,생활용품 상품 022,생활용품,3248000,29,8,8
86,89,생활용품 상품 089,생활용품,3090000,30,11,11


## 54. 주문 날짜 변환과 주문 월 생성

In [29]:
completed_items["order_date"] = pd.to_datetime(
    completed_items["order_date"],
    errors="coerce",
)
print(
    "날짜 변환 실패:",
    completed_items["order_date"].isna().sum(),

)

날짜 변환 실패: 0


In [30]:
completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)

## 55. 월별 매출

In [31]:
monthly_sales = (
    completed_items
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values("order_month")
)

display(monthly_sales)

,order_month,total_sales,order_count,customer_count,quantity_sold
0,2025-08,6582000,9,9,59
1,2025-09,16118000,19,19,131
2,2025-10,12895000,15,15,126
3,2025-11,23611000,25,24,235
4,2025-12,8621000,11,11,85
5,2026-01,10935000,14,14,106
6,2026-02,17154000,21,19,155
7,2026-03,9151000,17,15,96
8,2026-04,15536000,17,16,157
9,2026-05,15402000,20,19,156


## 56. 고객별 구매 금액

In [32]:

customer_sales = (
    completed_items
    .groupby("customer_id", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
)

In [33]:
print(customer_sales.sort_values("total_sales", ascending=False).head(10))

    customer_id  total_sales  order_count  quantity_sold
76        117.0      4100000            5             48
62        102.0      3996000            4             35
51         83.0      3880000            4             39
21         30.0      3590000            5             32
29         40.0      3523000            4             27
13         20.0      3191000            2             25
0           3.0      3178000            2             26
70        111.0      3153000            3             38
42         66.0      3093000            4             30
97        147.0      2990000            2             21


## 57. 고객 속성 연결

In [34]:
customer_attributes = customers[
    ["customer_id", "gender", "age", "city"]
].copy()

In [35]:
customer_sales_detail = (
    customer_sales
    .merge(
        customer_attributes,
        on="customer_id",
        how="left",
        validate="one_to_one",
        indicator="customer_match",
    )
    .sort_values("total_sales", ascending=False)
)

In [36]:
display(
    customer_sales_detail["customer_match"].value_counts(
        dropna=False
    )
)
display(customer_sales_detail.head(10))

customer_match
both          100
left_only       0
right_only      0
Name: count, dtype: int64

,customer_id,total_sales,order_count,quantity_sold,gender,age,city,customer_match
76,117.0,4100000,5,48,F,65,성남,both
62,102.0,3996000,4,35,M,60,고양,both
51,83.0,3880000,4,39,F,22,수원,both
21,30.0,3590000,5,32,F,32,서울,both
29,40.0,3523000,4,27,M,23,서울,both
13,20.0,3191000,2,25,F,20,인천,both
0,3.0,3178000,2,26,F,61,성남,both
70,111.0,3153000,3,38,F,41,광주,both
42,66.0,3093000,4,30,F,39,서울,both
97,147.0,2990000,2,21,M,19,부산,both


## 58. 결과 폴더 생성

In [37]:
output_dir = project_root / "reports" / "chapter04"
output_dir.mkdir(parents=True, exist_ok=True)
print(output_dir)

c:\Dev\ai-data-analysis\reports\chapter04


## 59. 결과 파일 저장

In [38]:
outputs = {
    "category_sales.csv": category_sales,
    "product_sales.csv": product_sales,
    "monthly_sales.csv": monthly_sales,
    "customer_sales.csv": customer_sales_detail,
}

for file_name, df in outputs.items():
    output_path = output_dir / file_name
    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(
        file_name,
        output_path.exists(),
        output_path.stat().st_size,
    )

category_sales.csv True 309
product_sales.csv True 4833
monthly_sales.csv True 413
customer_sales.csv True 3648


## 60. 저장 결과 다시 읽기

In [39]:
saved_category_sales = pd.read_csv(
    output_dir / "category_sales.csv"
)
display(saved_category_sales.head())
print(saved_category_sales.shape)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
0,스포츠,31743000,85,67,295,100
1,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
3,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42


(7, 6)


## 61. 병합 점검 함수

In [40]:
def check_merge_result(
    *,
    name: str,
    left_rows: int,
    merged: pd.DataFrame,
    indicator_column: str,
) -> None:
    print(f"[{name}]")
    print("병합 전 행 수:", left_rows)
    print("병합 후 행 수:", len(merged))
    print(
        merged[indicator_column].value_counts(
            dropna=False
        )
    )

In [41]:
# 함수 호출
check_merge_result(
    name="주문상세-주문",
    left_rows=len(order_items_work),
    merged=order_sales,
    indicator_column="order_match",
)

[주문상세-주문]
병합 전 행 수: 765
병합 후 행 수: 765
order_match
both          764
left_only       1
right_only      0
Name: count, dtype: int64


## 62. 집계 합계 검증 함수

In [42]:
def check_total(
    *,
    name: str,
    source_total: float,
    summary_total: float,
) -> None:
    difference = source_total - summary_total
    print(f"[{name}]")
    print("원본 합계:", source_total)
    print("요약 합계:", summary_total)
    print("차이:", difference)

In [43]:
check_total(
    name="카테고리별 매출",
    source_total=completed_items["line_total"].sum(),
    summary_total=category_sales["total_sales"].sum(),
)

[카테고리별 매출]
원본 합계: 148990000
요약 합계: 148990000
차이: 0


## LLM 검증표

| 검증 항목         | 확인 내용                      | 결과                                                                                                             |
| ------------- | -------------------------- | -------------------------------------------------------------------------------------------------------------- |
| **DataFrame** | 실제 변수명과 같은가?               | ✅ `orders`, `order_items`, `products` 실제 변수명을 사용함                                                              |
| **컬럼**        | 실제 컬럼만 사용하는가?              | ✅ 실제 존재하는 `order_id`, `customer_id`, `product_id`, `quantity`, `unit_price`, `order_status`, `category` 등을 사용함 |
| **상태값**       | `completed` 표기가 맞는가?       | ✅ 실제 `order_status`에 `completed`가 존재하며 `order_status == "completed"`로 필터링함                                     |
| **계산식**       | `quantity × unit_price`인가? | ✅ `line_total = quantity * unit_price`로 계산함. 수작업 결과와 실제 값도 일치                                                  |
| **분석 범위**     | 완료 주문만 포함하는가?              | ✅ `order_status == "completed"`인 주문만 `completed_sales`로 추출함                                                    |
| **주문 수**      | `nunique()`를 사용하는가?        | ✅ `order_id.nunique()`를 사용하여 중복 주문을 하나의 주문으로 계산함                                                               |
| **병합 키**      | 실제 관계와 맞는가?                | ✅ `order_items.order_id → orders.order_id`, `order_items.product_id → products.product_id`로 병합함                |
| **validate**  | `many_to_one`이 적용되었는가?     | ✅ 주문 병합과 상품 병합 모두 `validate="many_to_one"` 적용                                                                  |
| **indicator** | 미매칭을 확인하는가?                | ✅ 두 merge 모두 `indicator`를 사용하여 미매칭 여부 확인                                                                       |
| **행 수**       | 병합 전후를 비교하는가?              | ✅ 주문 병합 `765 → 765`, 상품 병합 `474 → 474`로 비교함                                                                    |
| **합계**        | 원본과 요약 합계를 비교하는가?          | ✅ 완료 주문 전체 매출과 카테고리별 매출 합계가 `148,990,000`으로 일치                                                                 |
| **개인정보**      | 원본 고객 정보를 요구하지 않는가?        | ✅ 고객 수 계산에는 `customer_id`만 사용하며 이름·성별·나이·도시 등 개인정보는 불필요                                                        |


orders 병합 전: 765행

orders 병합 후: 765행

both: 764

left_only: 1

right_only: 0

즉, 주문 1건이 매칭되지 않았습니다.